In [1]:
# installing everything we need for this project
import subprocess
import sys

packages = [
    'sqlalchemy',
    'prophet',
    'plotly',
    'mlxtend',
    'scipy',
    'statsmodels'
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    print(f"installed {package}")

print("done! all good to go")

installed sqlalchemy
installed prophet
installed plotly
installed mlxtend
installed scipy
installed statsmodels
done! all good to go


In [2]:
# importing everything we need
import pandas as pd
import numpy as np
import sqlite3
from sqlalchemy import create_engine, text
import logging
import os
import warnings
warnings.filterwarnings('ignore')

# setting up logging so we can track every step
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# creating outputs folder if it doesn't exist
os.makedirs('outputs', exist_ok=True)
os.makedirs('data', exist_ok=True)

print("all imports successful!")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
logger.info("environment setup complete")

2026-07-21 20:29:50,521 - INFO - environment setup complete


all imports successful!
pandas version: 3.0.3
numpy version: 2.5.1


In [3]:
# loading all 4 raw data files
logger.info("starting data ingestion...")

# load products table
products = pd.read_csv('annex1.csv')
logger.info(f"products loaded: {products.shape[0]} rows, {products.shape[1]} columns")

# load sales transactions - this is the big one
sales = pd.read_csv('annex2.csv')
logger.info(f"sales loaded: {sales.shape[0]} rows, {sales.shape[1]} columns")

# load product cost history
cost = pd.read_csv('annex3.csv')
logger.info(f"product cost loaded: {cost.shape[0]} rows, {cost.shape[1]} columns")

# load product loss rate
loss = pd.read_csv('annex4.csv')
logger.info(f"loss rate loaded: {loss.shape[0]} rows, {loss.shape[1]} columns")

print("\n--- DATA INGESTION SUMMARY ---")
print(f"products table:      {products.shape[0]:,} rows")
print(f"sales transactions:  {sales.shape[0]:,} rows")
print(f"product cost:        {cost.shape[0]:,} rows")
print(f"product loss rate:   {loss.shape[0]:,} rows")
print(f"\ntotal records ingested: {products.shape[0] + sales.shape[0] + cost.shape[0] + loss.shape[0]:,}")

2026-07-21 20:29:50,524 - INFO - starting data ingestion...
2026-07-21 20:29:50,540 - INFO - products loaded: 251 rows, 4 columns
2026-07-21 20:29:51,338 - INFO - sales loaded: 878503 rows, 7 columns
2026-07-21 20:29:51,381 - INFO - product cost loaded: 55982 rows, 3 columns
2026-07-21 20:29:51,385 - INFO - loss rate loaded: 251 rows, 3 columns



--- DATA INGESTION SUMMARY ---
products table:      251 rows
sales transactions:  878,503 rows
product cost:        55,982 rows
product loss rate:   251 rows

total records ingested: 934,987


In [4]:
# data quality checks - what a real data engineer does first
logger.info("running data quality checks...")

def check_data_quality(df, name):
    print(f"\n--- {name.upper()} QUALITY REPORT ---")
    print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"\nmissing values:")
    missing = df.isnull().sum()
    for col, count in missing.items():
        pct = (count / len(df)) * 100
        status = "WARNING" if count > 0 else "OK"
        print(f"  {col}: {count:,} missing ({pct:.2f}%) - {status}")
    print(f"\nduplicates: {df.duplicated().sum():,}")
    print(f"\ndata types:")
    for col, dtype in df.dtypes.items():
        print(f"  {col}: {dtype}")

check_data_quality(products, "products")
check_data_quality(sales, "sales transactions")
check_data_quality(cost, "product cost")
check_data_quality(loss, "product loss rate")

logger.info("data quality checks complete")

2026-07-21 20:29:51,397 - INFO - running data quality checks...



--- PRODUCTS QUALITY REPORT ---
shape: 251 rows x 4 columns

missing values:
  Item Code: 0 missing (0.00%) - OK
  Item Name: 0 missing (0.00%) - OK
  Category Code: 0 missing (0.00%) - OK
  Category Name: 0 missing (0.00%) - OK

duplicates: 0

data types:
  Item Code: int64
  Item Name: str
  Category Code: int64
  Category Name: str

--- SALES TRANSACTIONS QUALITY REPORT ---
shape: 878,503 rows x 7 columns

missing values:
  Date: 0 missing (0.00%) - OK
  Time: 0 missing (0.00%) - OK
  Item Code: 0 missing (0.00%) - OK
  Quantity Sold (kilo): 0 missing (0.00%) - OK
  Unit Selling Price (RMB/kg): 0 missing (0.00%) - OK
  Sale or Return: 0 missing (0.00%) - OK
  Discount (Yes/No): 0 missing (0.00%) - OK


2026-07-21 20:29:52,825 - INFO - data quality checks complete



duplicates: 0

data types:
  Date: str
  Time: str
  Item Code: int64
  Quantity Sold (kilo): float64
  Unit Selling Price (RMB/kg): float64
  Sale or Return: str
  Discount (Yes/No): str

--- PRODUCT COST QUALITY REPORT ---
shape: 55,982 rows x 3 columns

missing values:
  Date: 0 missing (0.00%) - OK
  Item Code: 0 missing (0.00%) - OK
  Wholesale Price (RMB/kg): 0 missing (0.00%) - OK

duplicates: 0

data types:
  Date: str
  Item Code: int64
  Wholesale Price (RMB/kg): float64

--- PRODUCT LOSS RATE QUALITY REPORT ---
shape: 251 rows x 3 columns

missing values:
  Item Code: 0 missing (0.00%) - OK
  Item Name: 0 missing (0.00%) - OK
  Loss Rate (%): 0 missing (0.00%) - OK

duplicates: 0

data types:
  Item Code: int64
  Item Name: str
  Loss Rate (%): float64


In [5]:
# data cleaning and transformation
logger.info("starting data cleaning and transformation...")

# fix date column in sales
sales['Date'] = pd.to_datetime(sales['Date'])
logger.info("converted sales date to datetime")

# fix date column in cost
cost['Date'] = pd.to_datetime(cost['Date'])
logger.info("converted cost date to datetime")

# extract useful time features from sales
sales['Year'] = sales['Date'].dt.year
sales['Month'] = sales['Date'].dt.month
sales['Month_Name'] = sales['Date'].dt.strftime('%B')
sales['Day_of_Week'] = sales['Date'].dt.day_name()
sales['Quarter'] = sales['Date'].dt.quarter
sales['Hour'] = pd.to_datetime(sales['Time'], format='%H:%M:%S.%f', errors='coerce').dt.hour

# clean up column names
sales.rename(columns={
    'Quantity Sold (kilo)': 'Quantity_Sold',
    'Unit Selling Price (RMB/kg)': 'Unit_Price_RMB',
    'Sale or Return': 'Transaction_Type',
    'Discount (Yes/No)': 'Discount'
}, inplace=True)

cost.rename(columns={
    'Wholesale Price (RMB/kg)': 'Wholesale_Price_RMB',
    'Item Code': 'Item_Code'
}, inplace=True)

products.rename(columns={
    'Item Code': 'Item_Code',
    'Item Name': 'Item_Name',
    'Category Code': 'Category_Code',
    'Category Name': 'Category_Name'
}, inplace=True)

loss.rename(columns={
    'Item Code': 'Item_Code',
    'Item Name': 'Item_Name',
    'Loss Rate (%)': 'Loss_Rate'
}, inplace=True)

sales.rename(columns={'Item Code': 'Item_Code'}, inplace=True)

# currency conversion RMB to USD
RMB_TO_USD = 0.14
sales['Unit_Price_USD'] = (sales['Unit_Price_RMB'] * RMB_TO_USD).round(4)
cost['Wholesale_Price_USD'] = (cost['Wholesale_Price_RMB'] * RMB_TO_USD).round(4)
logger.info(f"currency conversion applied: 1 RMB = {RMB_TO_USD} USD")

print("--- TRANSFORMATION SUMMARY ---")
print(f"date columns converted to datetime - OK")
print(f"currency conversion applied (RMB to USD) - OK")
print(f"column names standardized - OK")
print(f"time features extracted: Year, Month, Quarter, Day_of_Week, Hour - OK")
print(f"\nsales date range: {sales['Date'].min()} to {sales['Date'].max()}")
print(f"years covered: {sorted(sales['Year'].unique())}")
print(f"transaction types: {sales['Transaction_Type'].unique()}")
print(f"discount values: {sales['Discount'].unique()}")

logger.info("data cleaning and transformation complete")

2026-07-21 20:29:52,879 - INFO - starting data cleaning and transformation...
2026-07-21 20:29:53,389 - INFO - converted sales date to datetime
2026-07-21 20:29:53,420 - INFO - converted cost date to datetime
2026-07-21 20:29:58,516 - INFO - currency conversion applied: 1 RMB = 0.14 USD
2026-07-21 20:29:58,545 - INFO - data cleaning and transformation complete


--- TRANSFORMATION SUMMARY ---
date columns converted to datetime - OK
currency conversion applied (RMB to USD) - OK
column names standardized - OK
time features extracted: Year, Month, Quarter, Day_of_Week, Hour - OK

sales date range: 2020-07-01 00:00:00 to 2023-06-30 00:00:00
years covered: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023)]
transaction types: <ArrowStringArray>
['sale', 'return']
Length: 2, dtype: str
discount values: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


In [6]:
# feature engineering - creating new business metrics
logger.info("starting feature engineering...")

# merge cost data into sales using date and item code
sales['Link_Key'] = sales['Date'].astype(str) + '_' + sales['Item_Code'].astype(str)
cost['Link_Key'] = cost['Date'].astype(str) + '_' + cost['Item_Code'].astype(str)

sales = sales.merge(cost[['Link_Key', 'Wholesale_Price_RMB', 'Wholesale_Price_USD']], 
                    on='Link_Key', how='left')

# merge products info
sales = sales.merge(products[['Item_Code', 'Item_Name', 'Category_Code', 'Category_Name']], 
                    on='Item_Code', how='left')

# merge loss rate
sales = sales.merge(loss[['Item_Code', 'Loss_Rate']], 
                    on='Item_Code', how='left')

# calculate business metrics
sales['Sales_Amount_RMB'] = (sales['Quantity_Sold'] * sales['Unit_Price_RMB']).round(4)
sales['Sales_Amount_USD'] = (sales['Quantity_Sold'] * sales['Unit_Price_USD']).round(4)
sales['Cost_Amount_USD'] = (sales['Quantity_Sold'] * sales['Wholesale_Price_USD']).round(4)
sales['Gross_Profit_USD'] = (sales['Sales_Amount_USD'] - sales['Cost_Amount_USD']).round(4)
sales['Profit_Margin'] = ((sales['Gross_Profit_USD'] / sales['Sales_Amount_USD']) * 100).round(2)
sales['Loss_Adjusted_Revenue'] = (sales['Sales_Amount_USD'] * (1 - sales['Loss_Rate'] / 100)).round(4)
sales['Is_Sale'] = (sales['Transaction_Type'] == 'sale').astype(int)
sales['Is_Return'] = (sales['Transaction_Type'] == 'return').astype(int)
sales['Is_Discounted'] = (sales['Discount'] == 'Yes').astype(int)

logger.info("feature engineering complete")

print("--- FEATURE ENGINEERING SUMMARY ---")
print(f"total columns after engineering: {sales.shape[1]}")
print(f"\nnew features created:")
print(f"  Sales_Amount_RMB - total revenue in chinese yuan")
print(f"  Sales_Amount_USD - total revenue in US dollars")
print(f"  Cost_Amount_USD - total cost in US dollars")
print(f"  Gross_Profit_USD - profit after cost")
print(f"  Profit_Margin - profit as percentage")
print(f"  Loss_Adjusted_Revenue - revenue after accounting for product loss")
print(f"  Is_Sale, Is_Return, Is_Discounted - binary flags")
print(f"\nsample of engineered data:")
print(sales[['Item_Name', 'Category_Name', 'Sales_Amount_USD', 
             'Gross_Profit_USD', 'Profit_Margin']].head())

2026-07-21 20:29:58,551 - INFO - starting feature engineering...
2026-07-21 20:29:59,411 - INFO - feature engineering complete


--- FEATURE ENGINEERING SUMMARY ---
total columns after engineering: 30

new features created:
  Sales_Amount_RMB - total revenue in chinese yuan
  Sales_Amount_USD - total revenue in US dollars
  Cost_Amount_USD - total cost in US dollars
  Gross_Profit_USD - profit after cost
  Profit_Margin - profit as percentage
  Loss_Adjusted_Revenue - revenue after accounting for product loss
  Is_Sale, Is_Return, Is_Discounted - binary flags

sample of engineered data:
              Item_Name           Category_Name  Sales_Amount_USD  \
0  Paopaojiao (Jingpin)                Capsicum            0.4213   
1       Chinese Cabbage  Flower/Leaf Vegetables            0.3804   
2  Paopaojiao (Jingpin)                Capsicum            0.4352   
3          Shanghaiqing  Flower/Leaf Vegetables            0.5894   
4                Caixin  Flower/Leaf Vegetables            0.6037   

   Gross_Profit_USD  Profit_Margin  
0            0.1818          43.15  
1            0.1308          34.38  
2        

In [7]:
# building the star schema database
logger.info("building star schema database...")

# create database connection
engine = create_engine('sqlite:///supermarket_sales.db')

# dimension table 1 - products
dim_products = products.copy()
dim_products.to_sql('dim_products', engine, if_exists='replace', index=False)
logger.info(f"dim_products table created: {len(dim_products)} rows")

# dimension table 2 - categories
dim_categories = products[['Category_Code', 'Category_Name']].drop_duplicates()
dim_categories.to_sql('dim_categories', engine, if_exists='replace', index=False)
logger.info(f"dim_categories table created: {len(dim_categories)} rows")

# dimension table 3 - loss rates
dim_loss = loss.copy()
dim_loss.to_sql('dim_loss_rate', engine, if_exists='replace', index=False)
logger.info(f"dim_loss_rate table created: {len(dim_loss)} rows")

# dimension table 4 - date dimension
dim_date = sales[['Date', 'Year', 'Month', 'Month_Name', 
                   'Quarter', 'Day_of_Week']].drop_duplicates()
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)
logger.info(f"dim_date table created: {len(dim_date)} rows")

# fact table - sales transactions
fact_sales = sales[[
    'Date', 'Item_Code', 'Category_Name', 'Item_Name',
    'Quantity_Sold', 'Unit_Price_RMB', 'Unit_Price_USD',
    'Wholesale_Price_RMB', 'Wholesale_Price_USD',
    'Sales_Amount_RMB', 'Sales_Amount_USD',
    'Cost_Amount_USD', 'Gross_Profit_USD', 'Profit_Margin',
    'Loss_Adjusted_Revenue', 'Loss_Rate',
    'Transaction_Type', 'Discount',
    'Is_Sale', 'Is_Return', 'Is_Discounted',
    'Year', 'Month', 'Month_Name', 'Quarter', 
    'Day_of_Week', 'Hour'
]]
fact_sales.to_sql('fact_sales', engine, if_exists='replace', index=False)
logger.info(f"fact_sales table created: {len(fact_sales):,} rows")

print("--- STAR SCHEMA DATABASE SUMMARY ---")
print(f"database: supermarket_sales.db")
print(f"\ndimension tables:")
print(f"  dim_products:    {len(dim_products)} rows")
print(f"  dim_categories:  {len(dim_categories)} rows")
print(f"  dim_loss_rate:   {len(dim_loss)} rows")
print(f"  dim_date:        {len(dim_date)} rows")
print(f"\nfact table:")
print(f"  fact_sales:      {len(fact_sales):,} rows")
print(f"\ndatabase saved to: supermarket_sales.db")

logger.info("star schema database built successfully")

2026-07-21 20:29:59,423 - INFO - building star schema database...
2026-07-21 20:29:59,452 - INFO - dim_products table created: 251 rows
2026-07-21 20:29:59,461 - INFO - dim_categories table created: 6 rows
2026-07-21 20:29:59,469 - INFO - dim_loss_rate table created: 251 rows
2026-07-21 20:29:59,583 - INFO - dim_date table created: 1085 rows
2026-07-21 20:30:20,482 - INFO - fact_sales table created: 878,503 rows
2026-07-21 20:30:20,483 - INFO - star schema database built successfully


--- STAR SCHEMA DATABASE SUMMARY ---
database: supermarket_sales.db

dimension tables:
  dim_products:    251 rows
  dim_categories:  6 rows
  dim_loss_rate:   251 rows
  dim_date:        1085 rows

fact table:
  fact_sales:      878,503 rows

database saved to: supermarket_sales.db


In [8]:
# advanced SQL queries - window functions, CTEs, aggregations
logger.info("running advanced SQL queries...")

conn = sqlite3.connect('supermarket_sales.db')

# SQL Query 1 - running total and rolling average using window functions
query1 = """
WITH monthly_sales AS (
    SELECT 
        Year,
        Month,
        Month_Name,
        ROUND(SUM(Sales_Amount_USD), 2) as total_sales,
        ROUND(SUM(Gross_Profit_USD), 2) as total_profit,
        COUNT(*) as total_transactions
    FROM fact_sales
    WHERE Transaction_Type = 'sale'
    GROUP BY Year, Month, Month_Name
)
SELECT 
    Year,
    Month,
    Month_Name,
    total_sales,
    total_profit,
    total_transactions,
    ROUND(SUM(total_sales) OVER (ORDER BY Year, Month), 2) as running_total_sales,
    ROUND(AVG(total_sales) OVER (ORDER BY Year, Month 
          ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) as rolling_3month_avg
FROM monthly_sales
ORDER BY Year, Month
"""

monthly_trends = pd.read_sql_query(query1, conn)
print("--- QUERY 1: MONTHLY SALES WITH RUNNING TOTALS ---")
print(monthly_trends.to_string(index=False))
logger.info("monthly trends query complete")

2026-07-21 20:30:20,487 - INFO - running advanced SQL queries...
2026-07-21 20:30:21,396 - INFO - monthly trends query complete


--- QUERY 1: MONTHLY SALES WITH RUNNING TOTALS ---
 Year  Month Month_Name  total_sales  total_profit  total_transactions  running_total_sales  rolling_3month_avg
 2020      7       July     17186.60       6876.44               33559             17186.60            17186.60
 2020      8     August     18750.53       7183.63               36577             35937.13            17968.56
 2020      9  September     14918.60       5772.08               28148             50855.73            16951.91
 2020     10    October     17181.06       6556.59               31876             68036.79            16950.06
 2020     11   November     11845.23       4234.67               28034             79882.02            14648.30
 2020     12   December     13919.67       4184.14               27560             93801.69            14315.32
 2021      1    January     19004.99       5739.76               28647            112806.68            14923.30
 2021      2   February     25042.68       8768.34   

In [9]:
# SQL Query 2 - top performing products using RANK window function
query2 = """
WITH product_performance AS (
    SELECT 
        Item_Name,
        Category_Name,
        COUNT(*) as total_transactions,
        ROUND(SUM(Quantity_Sold), 2) as total_qty_sold,
        ROUND(SUM(Sales_Amount_USD), 2) as total_revenue,
        ROUND(SUM(Gross_Profit_USD), 2) as total_profit,
        ROUND(AVG(Profit_Margin), 2) as avg_profit_margin,
        ROUND(SUM(CASE WHEN Is_Return = 1 THEN Quantity_Sold ELSE 0 END) * 100.0 
              / SUM(Quantity_Sold), 2) as return_rate_pct,
        ROUND(SUM(CASE WHEN Is_Discounted = 1 THEN Sales_Amount_USD ELSE 0 END) * 100.0 
              / SUM(Sales_Amount_USD), 2) as discount_rate_pct
    FROM fact_sales
    GROUP BY Item_Name, Category_Name
),
ranked_products AS (
    SELECT *,
        RANK() OVER (ORDER BY total_revenue DESC) as revenue_rank,
        RANK() OVER (ORDER BY total_profit DESC) as profit_rank,
        RANK() OVER (ORDER BY return_rate_pct ASC) as quality_rank
    FROM product_performance
)
SELECT 
    revenue_rank,
    Item_Name,
    Category_Name,
    total_transactions,
    total_revenue,
    total_profit,
    avg_profit_margin,
    return_rate_pct,
    discount_rate_pct
FROM ranked_products
WHERE revenue_rank <= 10
ORDER BY revenue_rank
"""

top_products = pd.read_sql_query(query2, conn)
print("--- QUERY 2: TOP 10 PRODUCTS BY REVENUE ---")
print(top_products.to_string(index=False))
logger.info("top products query complete")

2026-07-21 20:30:23,003 - INFO - top products query complete


--- QUERY 2: TOP 10 PRODUCTS BY REVENUE ---
 revenue_rank             Item_Name               Category_Name  total_transactions  total_revenue  total_profit  avg_profit_margin  return_rate_pct  discount_rate_pct
            1              Broccoli                     Cabbage               58905       37782.36      12618.80              33.00            -0.07               3.75
            2    Net Lotus Root (1) Aquatic Tuberous Vegetables               39285       29631.28       8727.62              28.85            -0.06               8.95
            3    Xixia Mushroom (1)             Edible Mushroom               47509       29567.83      10005.94              33.42            -0.07               9.22
            4 Wuhu Green Pepper (1)                    Capsicum               69945       28715.87       8595.05              29.63            -0.06               1.83
            5       Yunnan Shengcai      Flower/Leaf Vegetables               39887       18166.02       6996.90    

In [10]:
# SQL Query 3 - discount effectiveness analysis
query3 = """
WITH discount_analysis AS (
    SELECT 
        Category_Name,
        Discount,
        COUNT(*) as transactions,
        ROUND(AVG(Quantity_Sold), 4) as avg_qty_sold,
        ROUND(AVG(Unit_Price_USD), 4) as avg_price,
        ROUND(SUM(Sales_Amount_USD), 2) as total_revenue,
        ROUND(AVG(Profit_Margin), 2) as avg_profit_margin,
        ROUND(SUM(Gross_Profit_USD), 2) as total_profit
    FROM fact_sales
    WHERE Transaction_Type = 'sale'
    GROUP BY Category_Name, Discount
),
pivot_analysis AS (
    SELECT 
        Category_Name,
        MAX(CASE WHEN Discount = 'Yes' THEN avg_profit_margin END) as discounted_margin,
        MAX(CASE WHEN Discount = 'No' THEN avg_profit_margin END) as full_price_margin,
        MAX(CASE WHEN Discount = 'Yes' THEN total_revenue END) as discounted_revenue,
        MAX(CASE WHEN Discount = 'No' THEN total_revenue END) as full_price_revenue,
        MAX(CASE WHEN Discount = 'Yes' THEN transactions END) as discounted_transactions,
        MAX(CASE WHEN Discount = 'No' THEN transactions END) as full_price_transactions
    FROM discount_analysis
    GROUP BY Category_Name
)
SELECT 
    Category_Name,
    ROUND(full_price_margin, 2) as full_price_margin_pct,
    ROUND(discounted_margin, 2) as discounted_margin_pct,
    ROUND(full_price_margin - discounted_margin, 2) as margin_loss_from_discount,
    full_price_transactions,
    discounted_transactions,
    ROUND(full_price_revenue, 2) as full_price_revenue,
    ROUND(discounted_revenue, 2) as discounted_revenue
FROM pivot_analysis
ORDER BY margin_loss_from_discount DESC
"""

discount_impact = pd.read_sql_query(query3, conn)
print("--- QUERY 3: DISCOUNT IMPACT BY CATEGORY ---")
print(discount_impact.to_string(index=False))
logger.info("discount analysis query complete")

2026-07-21 20:30:24,207 - INFO - discount analysis query complete


--- QUERY 3: DISCOUNT IMPACT BY CATEGORY ---
              Category_Name  full_price_margin_pct  discounted_margin_pct  margin_loss_from_discount  full_price_transactions  discounted_transactions  full_price_revenue  discounted_revenue
     Flower/Leaf Vegetables                  40.84                  16.80                      24.04                   315108                    16681           144574.24             6602.54
            Edible Mushroom                  37.36                  14.75                      22.61                   136647                    11694            81153.26             5662.24
                    Cabbage                  35.59                  15.59                      20.00                    81879                     4645            50260.48             2376.81
                   Capsicum                  36.97                  18.85                      18.12                   200501                     7393           102056.21             3582.89


In [11]:
# SQL Query 4 - peak hours and day of week analysis
query4 = """
WITH hourly_sales AS (
    SELECT 
        Hour,
        Day_of_Week,
        COUNT(*) as transactions,
        ROUND(SUM(Sales_Amount_USD), 2) as total_revenue,
        ROUND(AVG(Sales_Amount_USD), 4) as avg_transaction_value,
        ROUND(SUM(Quantity_Sold), 2) as total_qty
    FROM fact_sales
    WHERE Transaction_Type = 'sale'
    AND Hour IS NOT NULL
    GROUP BY Hour, Day_of_Week
),
hour_totals AS (
    SELECT 
        Hour,
        SUM(transactions) as total_transactions,
        ROUND(SUM(total_revenue), 2) as total_revenue,
        ROUND(AVG(avg_transaction_value), 4) as avg_transaction_value,
        RANK() OVER (ORDER BY SUM(total_revenue) DESC) as revenue_rank
    FROM hourly_sales
    GROUP BY Hour
)
SELECT 
    Hour,
    total_transactions,
    total_revenue,
    avg_transaction_value,
    revenue_rank,
    CASE 
        WHEN Hour BETWEEN 6 AND 9 THEN 'Morning Rush'
        WHEN Hour BETWEEN 10 AND 12 THEN 'Mid Morning'
        WHEN Hour BETWEEN 13 AND 15 THEN 'Afternoon'
        WHEN Hour BETWEEN 16 AND 19 THEN 'Evening Rush'
        ELSE 'Off Peak'
    END as time_period
FROM hour_totals
ORDER BY Hour
"""

hourly_analysis = pd.read_sql_query(query4, conn)
print("--- QUERY 4: PEAK HOURS ANALYSIS ---")
print(hourly_analysis.to_string(index=False))
logger.info("peak hours analysis complete")

2026-07-21 20:30:25,010 - INFO - peak hours analysis complete


--- QUERY 4: PEAK HOURS ANALYSIS ---
 Hour  total_transactions  total_revenue  avg_transaction_value  revenue_rank  time_period
  8.0                   1           0.55                 0.5474            15 Morning Rush
  9.0               74021       39423.35                 0.5323             6 Morning Rush
 10.0              120870       65590.77                 0.5424             1  Mid Morning
 11.0               94688       51544.70                 0.5456             2  Mid Morning
 12.0               49142       27013.93                 0.5508            10  Mid Morning
 13.0               35564       20012.76                 0.5649            12    Afternoon
 14.0               41755       23696.05                 0.5693            11    Afternoon
 15.0               62072       34623.47                 0.5587             7    Afternoon
 16.0               82702       44846.69                 0.5427             5 Evening Rush
 17.0               92736       49390.71             

In [12]:
# SQL Query 5 - year over year performance by category
query5 = """
WITH yearly_category AS (
    SELECT 
        Year,
        Category_Name,
        COUNT(*) as transactions,
        ROUND(SUM(Sales_Amount_USD), 2) as total_revenue,
        ROUND(SUM(Gross_Profit_USD), 2) as total_profit,
        ROUND(AVG(Profit_Margin), 2) as avg_margin,
        ROUND(SUM(Quantity_Sold), 2) as total_qty
    FROM fact_sales
    WHERE Transaction_Type = 'sale'
    GROUP BY Year, Category_Name
),
yoy_comparison AS (
    SELECT 
        curr.Year,
        curr.Category_Name,
        curr.total_revenue as current_revenue,
        prev.total_revenue as previous_revenue,
        ROUND((curr.total_revenue - prev.total_revenue) 
              * 100.0 / prev.total_revenue, 2) as yoy_growth_pct,
        curr.avg_margin,
        curr.transactions,
        curr.total_profit
    FROM yearly_category curr
    LEFT JOIN yearly_category prev 
        ON curr.Category_Name = prev.Category_Name 
        AND curr.Year = prev.Year + 1
)
SELECT 
    Year,
    Category_Name,
    current_revenue,
    previous_revenue,
    yoy_growth_pct,
    avg_margin,
    transactions,
    total_profit
FROM yoy_comparison
ORDER BY Year, current_revenue DESC
"""

yoy_analysis = pd.read_sql_query(query5, conn)
print("--- QUERY 5: YEAR OVER YEAR CATEGORY PERFORMANCE ---")
print(yoy_analysis.to_string(index=False))
logger.info("year over year analysis complete")

2026-07-21 20:30:25,791 - INFO - year over year analysis complete


--- QUERY 5: YEAR OVER YEAR CATEGORY PERFORMANCE ---
 Year               Category_Name  current_revenue  previous_revenue  yoy_growth_pct  avg_margin  transactions  total_profit
 2020      Flower/Leaf Vegetables         32851.05               NaN             NaN       39.54         72363      13219.26
 2020             Edible Mushroom         19362.78               NaN             NaN       37.89         34598       7386.35
 2020                    Capsicum         17581.67               NaN             NaN       35.09         40445       6074.97
 2020                     Cabbage         11788.05               NaN             NaN       35.32         19297       4231.72
 2020 Aquatic Tuberous Vegetables          7373.24               NaN             NaN       28.79         10180       2133.44
 2020                     Solanum          4844.89               NaN             NaN       36.15          8871       1761.82
 2021      Flower/Leaf Vegetables         51123.17          32851.05    

In [13]:
# save all SQL query results to CSV for later use
logger.info("saving SQL query results...")

monthly_trends.to_csv('outputs/monthly_trends.csv', index=False)
top_products.to_csv('outputs/top_products.csv', index=False)
discount_impact.to_csv('outputs/discount_impact.csv', index=False)
hourly_analysis.to_csv('outputs/hourly_analysis.csv', index=False)
yoy_analysis.to_csv('outputs/yoy_analysis.csv', index=False)

# save the full cleaned dataset
sales.to_csv('outputs/fact_sales_cleaned.csv', index=False)

conn.close()

print("--- ALL SQL RESULTS SAVED ---")
print("monthly_trends.csv - OK")
print("top_products.csv - OK")
print("discount_impact.csv - OK")
print("hourly_analysis.csv - OK")
print("yoy_analysis.csv - OK")
print("fact_sales_cleaned.csv - OK")
print(f"\nETL pipeline complete!")
print(f"total records processed: {len(sales):,}")
print(f"database: supermarket_sales.db")
print(f"outputs saved to: outputs/")

logger.info("ETL pipeline complete - all outputs saved")

2026-07-21 20:30:25,794 - INFO - saving SQL query results...
2026-07-21 20:30:33,923 - INFO - ETL pipeline complete - all outputs saved


--- ALL SQL RESULTS SAVED ---
monthly_trends.csv - OK
top_products.csv - OK
discount_impact.csv - OK
hourly_analysis.csv - OK
yoy_analysis.csv - OK
fact_sales_cleaned.csv - OK

ETL pipeline complete!
total records processed: 878,503
database: supermarket_sales.db
outputs saved to: outputs/
